# 22 Where voids meet: boundaries through open void

The distance basins (notebooks 17 and 18) split the void voxels into voids by climbing the distance to structure. Where two basins meet in open void, the boundary is a geometric cut, roughly equidistant from two stretches of wall, not a wall itself; at Δ = 0.5 h⁻¹ Mpc such cuts show up as straight lines in slices. This notebook measures how much of that there is at each Δ, for the truth and for DeepVoid's predictions, before trying a better way to split.

**What is measured.** For the voids at a given Δ (minimum radius as notebook 18):

- **Internal boundary**: void voxels with a neighbor in a different void, as opposed to the **structure boundary** (a neighbor that is wall, filament or halo). Its share of all boundary faces says how much of the void surfaces are cuts between voids at all.
- **How far internal-boundary voxels sit from structure** (the distance to structure on them), in cells: ≤ 1 cell means the cut touches structure (a real wall or a thin neck); beyond 2 cells it runs through open void.
- **Per void**: the share of each void's surface that is an open-void cut (internal boundary more than 2 cells from structure), and how many voids above R_eff = 3 h⁻¹ Mpc have more than 10% and 25% of their surface that way.

A Δ floor (merging every cut shallower than a cell or two) is the same as a larger Δ, so the rows for Δ = 1, 2, 3 show what a floor would do.

**Two ways to cut the merge tree.** By depth, Δ (a length): merge where the shallower void's center is less than Δ deeper than the neck (`child_peak - pass <= Δ`). By constriction, r (dimensionless): merge where the neck is at least r times as wide as the shallower void's center (`pass >= r * child_peak`), i.e. keep two voids apart only where the neck pinches below r. The first run (Δ only) found that at Δ = 1 about half the boundaries between voids cross a neck almost as wide as the void center (pass > 0.7 peak): the straight cuts. The constriction cut targets those directly and does not depend on the cell or the tracer spacing. For each cut, the boundaries that survive are also described by their pass / peak.

**Output.** `void_definitions/void_boundaries.json` (the tables, reused on reruns) and the figures `void_boundaries.png`, `void_boundaries_slice.png`.

**Runtime.** CPU, High-RAM. For each source (truth, DeepVoid at λ = 3 and 10): the distance to structure, the unmerged basins and the merge tree at 512³, read from the shared cache (`voids/cache/`, the same files notebook 18 uses) when they exist and computed and saved once otherwise; then a quick cut per Δ and r.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

## 1 Settings

In [ ]:
import gc
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.ticker import MaxNLocator

from deepvoid import Catalog
from deepvoid import voids as V
from deepvoid.labels import VOID

RUNS = Path('/content/drive/MyDrive/deepvoid_runs')
OUT = RUNS / 'void_definitions'
OUT.mkdir(parents=True, exist_ok=True)
PRED = RUNS / 'predictions'                               # whole-box scores (notebook 17)
RESULT = OUT / 'void_boundaries.json'
TIMINGS = RUNS / 'voids' / 'timings.json'                 # shared with notebooks 17, 18, 21
CACHE = RUNS / 'voids' / 'cache'                          # notebook 17/18's cache: distance fields, unmerged basins and merge trees are shared
CACHE_PREFIX = 'v1_tng_th0.65_c1_'                        # notebook 18's names for TNG at λ_th = 0.65 (CACHE_VERSION 1, COARSEN 1)
USE_CACHE = True
catalog = Catalog()

LABELS = 'tng300-3/mask'                                  # TNG300-3, 512^3, 0.40 h^-1 Mpc
MODEL = 'subhalo_C_mixed_smooth2ch_depth5_distill'        # as notebook 18
SPACINGS = [3, 10]                                        # DeepVoid predictions to include (if saved)
DELTAS = [0.5, 1.0, 2.0, 3.0]                             # h^-1 Mpc: cut by depth (Δ)
CONSTRICTIONS = [0.5, 0.55, 0.6, 0.65, 0.7, 0.8]          # cut by constriction: merge where pass >= r × peak of the shallower void
                                                          # (truth voids above R_CUT: r = 0.8 / 0.65 / 0.55 give about as many as Δ = 1 / 2 / 3)
PINCHED = 0.7                                             # a surviving boundary with pass > this × peak is 'barely pinched'
SIGMA_EFF = 1.0                                           # h^-1 Mpc: T-web smoothing of the labels
MIN_RADIUS = SIGMA_EFF                                    # as notebook 18: the label smoothing
MARGIN = 96
R_CUT = 3.0                                               # voids counted per void: R_eff above this
OPEN_CELLS = 2.0                                          # an internal boundary farther than this from structure is an open-void cut
DIST_BINS = [0.0, 1.01, 2.01, 4.01, np.inf]               # cells (edges just above 1, 2, 4: a voxel next to structure is at exactly 1)
DIST_LABELS = ['1 (touches structure)', '1–2', '2–4', '> 4']
SLICE = 256                                               # x index of the figure slice
SLICE_CUTS = ['Δ = 1.0', 'r = 0.8', 'Δ = 2.0', 'r = 0.65']   # truth slices drawn, in pairs with about the same number of voids (when the truth is computed in this run)
REFRESH = False


def timed(name, fn):
    # fn(), with how long the same step took last time (if known); records this run's time
    t = json.loads(TIMINGS.read_text()) if TIMINGS.exists() else {}
    print(f'{name}: ' + (f'last time {t[name] / 60:.1f} min' if name in t else 'no earlier timing'), flush=True)
    t0 = time.time()
    res = fn()
    dt = time.time() - t0
    t = json.loads(TIMINGS.read_text()) if TIMINGS.exists() else {}
    t[name] = dt
    TIMINGS.write_text(json.dumps(t, indent=1))
    print(f'{name}: {dt / 60:.1f} min', flush=True)
    return res


def cached(name, fn):
    # As notebook 18: fn()'s result saved in the shared cache the first time and read back afterwards
    path = CACHE / f'{CACHE_PREFIX}{name}.npz'
    if USE_CACHE and path.exists():
        with np.load(path) as f:
            out = {k: f[k] for k in f.files}
        out = {k: v.astype(np.int32) if v.dtype == np.uint16 else v for k, v in out.items()}
        print(f'{name}: from the cache', flush=True)
        return out['_'] if list(out) == ['_'] else out
    res = timed(name, fn)
    d = res if isinstance(res, dict) else {'_': res}
    d = {k: v.astype(np.uint16) if (np.issubdtype(np.asarray(v).dtype, np.integer) and np.asarray(v).ndim == 3
                                    and 0 <= v.min() and v.max() < 65535) else v for k, v in d.items()}
    if USE_CACHE:
        CACHE.mkdir(parents=True, exist_ok=True)
        np.savez(path, **d)
    return res

## 2 Measure

For each source and cut: the voids, the two kinds of boundary, the distances to structure on the internal boundary, per-void shares, and the pass / peak of the boundaries that survive. Results go to `void_boundaries.json` after each source; a rerun computes only the cuts a source is missing or that lack the pass / peak columns (the first run's; the distance field and basins are rebuilt for that, a few minutes per source). Slices of the truth are kept in memory only, so they are drawn when the truth is computed in this run (`REFRESH = True` redraws them).

In [ ]:
labels = catalog.load(LABELS)
n, cell = labels.n, labels.box_size / labels.n
sources = {'T-web truth': lambda: labels.data == VOID}
tags = {'T-web truth': 'truth'}                             # cache names, as notebook 18
for s in SPACINGS:
    f = PRED / f'tng_{MODEL}_L{s}.npz'
    if f.exists():
        sources[f'DeepVoid λ = {s}'] = (lambda f=f: np.load(f)['scores'].argmax(-1) == VOID)
        tags[f'DeepVoid λ = {s}'] = f'{MODEL}_L{s}'
    else:
        print(f'{f.name} not found: run notebook 17 with TRUTH_ONLY = False; λ = {s} left out')

CUTS = {**{f'Δ = {d}': {'persistence': d} for d in DELTAS},
        **{f'r = {r}': {'constriction': r} for r in CONSTRICTIONS}}
OLD_KEYS = {f'Δ = {d}': str(d) for d in DELTAS}               # keys of the first run (Δ only)
results = json.loads(RESULT.read_text()) if RESULT.exists() and not REFRESH else {}
for src in results:                                         # first-run keys to the new ones
    results[src] = {next((k for k, o in OLD_KEYS.items() if o == key), key): v for key, v in results[src].items()}
slices = {}


def measure(name, todo):
    void = sources[name]()
    tag = tags[name]
    d = cached(f'{tag}_distance', lambda: V.distance_to_structure(void, cell=cell, margin=MARGIN))
    seeds = cached(f'{tag}_seed_basins', lambda: V.ascent_basins(d, void)[0])
    tree = cached(f'{tag}_merge_tree', lambda: V.merge_tree(seeds, d))
    ratio = tree['pass_'] / np.maximum(tree['child_peak'], 1e-9)
    rows = {}
    for cut in todo:
        merged = V.merge_mask(tree, **CUTS[cut])
        lab = V.cut_tree(seeds, tree, distance=d, min_radius=MIN_RADIUS, **CUTS[cut])
        b = V.internal_boundaries(lab, void)
        hist = np.histogram(d[b['internal']] / cell, DIST_BINS)[0]
        open_cut = b['internal'] & (d > (OPEN_CELLS + 0.01) * cell)
        n_lab = int(lab.max())
        surface = np.bincount(lab[b['internal'] | b['structure']], minlength=n_lab + 1)
        cutv = np.bincount(lab[open_cut], minlength=n_lab + 1)
        r_eff = V.void_catalog(lab, cell)['r_eff']
        big = np.flatnonzero(r_eff > R_CUT) + 1          # void labels are 1..n
        share = cutv[big] / np.maximum(surface[big], 1)
        surv = ratio[V.surviving_boundaries(tree, merged)]
        rows[cut] = {
            'voids above R_CUT': int(len(big)),
            'median R_eff': float(np.median(r_eff[r_eff > R_CUT])) if len(big) else float('nan'),
            'internal share of boundary faces': b['internal_faces'] / max(b['internal_faces'] + b['structure_faces'], 1),
            **{f'internal boundary at {DIST_LABELS[i]} cells': (float(h / hist.sum()) if hist.sum() else float('nan'))
               for i, h in enumerate(hist)},
            'median open-cut share of a void surface': float(np.median(share)) if len(share) else float('nan'),
            'voids with > 10% open-cut surface': float((share > 0.10).mean()) if len(share) else float('nan'),
            'voids with > 25% open-cut surface': float((share > 0.25).mean()) if len(share) else float('nan'),
            'boundaries between voids (tree)': int(len(surv)),
            'median pass / peak of boundaries': float(np.median(surv)) if len(surv) else float('nan'),
            f'boundaries with pass > {PINCHED:g} peak': float((surv > PINCHED).mean()) if len(surv) else float('nan'),
        }
        if name == 'T-web truth' and cut in SLICE_CUTS:
            slices[cut] = {'lab': lab[SLICE].copy(), 'internal': b['internal'][SLICE].copy(), 'd': d[SLICE].copy()}
        del lab, b, open_cut
    del void, d, seeds, tree
    gc.collect()
    return rows


for name in sources:
    todo = [c for c in CUTS if 'median pass / peak of boundaries' not in results.get(name, {}).get(c, {})]   # missing, or from the first run
    if not todo:
        print(f'{name}: from {RESULT.name}')
        continue
    rows = measure(name, todo)
    results.setdefault(name, {}).update(rows)
    RESULT.write_text(json.dumps(results, indent=1, ensure_ascii=False))

table = pd.DataFrame({(src, c): results[src][c] for src in results for c in CUTS if c in results[src]}).T
print(table.round(3).to_string())

## 3 Figures

Top left: share of the internal boundary at each distance from structure, per depth cut Δ (truth solid, predictions dashed / dotted). Top right: for every cut, the share of voids (R_eff > 3) with over 10% of their surface cut through open void, against the number of voids; a better cut sits lower at the same number of voids. Bottom: truth slices for the cuts in `SLICE_CUTS`, each void in its own color, structure in white, internal boundaries black where they touch structure (≤ 1 cell) and red where they run through open void (> 2 cells). Boundaries are 3D, so a void whose neighbor lies just above or below the slice shows red inside it, not only along the edges seen in the slice.

In [ ]:
srcs = list(results)
styles = {s: ('-' if s == 'T-web truth' else '--' if s.endswith('= 3') else ':') for s in srcs}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
colors = dict(zip(DELTAS, ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']))
for s in srcs:
    for dl in DELTAS:
        r = results[s].get(f'Δ = {dl}')
        if r:
            axes[0].plot(range(len(DIST_LABELS)), [r[f'internal boundary at {b} cells'] for b in DIST_LABELS],
                         styles[s], color=colors[dl], marker='o', ms=3)
    for kind, mk, col in (('Δ', 'o', 'k'), ('r', 's', 'tab:purple')):
        keys = [c for c in CUTS if c.startswith(kind) and c in results[s]]
        xs = [results[s][c]['voids above R_CUT'] for c in keys]
        ys = [results[s][c]['voids with > 10% open-cut surface'] for c in keys]
        axes[1].plot(xs, ys, styles[s], color=col, marker=mk, ms=5)
        if s == 'T-web truth':
            for c, x, y in zip(keys, xs, ys):
                axes[1].annotate(c, (x, y), textcoords='offset points', xytext=(4, 4), fontsize=8, color=col)
axes[0].set_xticks(range(len(DIST_LABELS)), DIST_LABELS, rotation=15)
axes[0].set_xlabel('distance from the internal-boundary voxel to structure (cells of 0.40 h$^{-1}$ Mpc)')
axes[0].set_ylabel('share of internal-boundary voxels')
axes[0].set_title('Where voids meet: distance from structure (depth cuts)')
axes[1].set_xlabel('number of voids (R$_{eff}$ > 3 h$^{-1}$ Mpc)')
axes[1].set_ylabel('share of voids')
axes[1].set_title('Voids with over 10% of their surface cut through open void')
for ax in axes:
    ax.set_ylim(bottom=0)
    ax.yaxis.set_major_locator(MaxNLocator(steps=[1, 2, 5, 10]))
axes[1].set_xlim(left=0)
handles = [plt.Line2D([], [], color=c, label=f'Δ = {dl} h$^{{-1}}$ Mpc (left panel)') for dl, c in colors.items()]
handles += [plt.Line2D([], [], color='k', marker='o', ls='none', label='depth cut Δ (right panel)'),
            plt.Line2D([], [], color='tab:purple', marker='s', ls='none', label='constriction cut r (right panel)')]
handles += [plt.Line2D([], [], color='gray', ls=styles[s], label=s) for s in srcs]
fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.0, 0.5), frameon=False)
fig.suptitle('Void boundaries between voids, TNG300-3 (512$^3$): how many run through open void')
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(OUT / 'void_boundaries.png', dpi=150, bbox_inches='tight')
plt.show()

if slices:
    rng = np.random.default_rng(0)
    shown = [c for c in SLICE_CUTS if c in slices]
    ncol = 2
    nrow = int(np.ceil(len(shown) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(7 * ncol, 7 * nrow), squeeze=False)
    ext = [0, n * cell, 0, n * cell]
    for ax, c in zip(axes.ravel(), shown):
        sl = slices[c]
        lab = sl['lab']
        cmap = ListedColormap(np.vstack([[1, 1, 1, 1], np.column_stack([rng.uniform(0.35, 0.9, (max(lab.max(), 1), 3)), np.ones(max(lab.max(), 1))])]))
        ax.imshow(lab.T, origin='lower', cmap=cmap, interpolation='nearest', extent=ext)
        touch = sl['internal'] & (sl['d'] <= 1.01 * cell)
        open_ = sl['internal'] & (sl['d'] > (OPEN_CELLS + 0.01) * cell)
        for m, col in ((touch, 'k'), (open_, 'red')):
            ov = np.zeros(m.shape + (4,))
            ov[m] = plt.matplotlib.colors.to_rgba(col)
            ax.imshow(ov.transpose(1, 0, 2), origin='lower', interpolation='nearest', extent=ext)
        nv = results['T-web truth'][c]['voids above R_CUT']
        ax.set_title(f'T-web truth, {c} ({nv} voids > 3 h$^{{-1}}$ Mpc), x = {SLICE * cell:.1f} h$^{{-1}}$ Mpc')
        ax.set_xlabel('y (h$^{-1}$ Mpc)')
        ax.set_ylabel('z (h$^{-1}$ Mpc)')
    for ax in axes.ravel()[len(shown):]:
        ax.axis('off')
    handles = [plt.Line2D([], [], color='k', lw=3, label='boundary between voids, touching structure (≤ 1 cell)'),
               plt.Line2D([], [], color='red', lw=3, label=f'boundary between voids through open void (> {OPEN_CELLS:g} cells)'),
               plt.Line2D([], [], color='lightgray', lw=6, label='one color per void; white = wall, filament, halo')]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.02), frameon=False, ncol=1)
    fig.suptitle('Truth voids in a slice: depth cut (Δ, h$^{-1}$ Mpc) against constriction cuts (r)')
    fig.tight_layout(rect=(0, 0.04, 1, 0.96))
    fig.savefig(OUT / 'void_boundaries_slice.png', dpi=150, bbox_inches='tight')
    plt.show()

## Shut down

Deletes this runtime once every cell above has finished, so it stops using compute units. The results are already on Drive. With **Run all**, Colab stops at the first cell that fails, so after an error the runtime stays up and you can look at it. Set `SHUT_DOWN = False` to keep the runtime.

In [ ]:
SHUT_DOWN = True   # False keeps the runtime after the run
if SHUT_DOWN:
    from google.colab import drive, runtime
    drive.flush_and_unmount()   # finish writing to Drive first
    time.sleep(5)
    runtime.unassign()          # disconnects and deletes this runtime